In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import sympy as sp
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100,'reso':'xx-hi'})


In [19]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SRCONFIG   = MODELS['sr']
SPLIT      = 'valid'
PLOTSTYLE  = {
    'pod':('gray6','POD'),
    'baseline':('blue6','Baseline NN'),
    'nonparametric':('yellow3','Nonparametric Kernel NN'),
    'parametric':('red6','Parametric Kernel NN'),
    'sr':('green7','Symbolic Regression')}

In [ ]:
def get_kind(name):
    for group in ['pod','nn','sr']:
        runs = MODELS.get(group,{}).get('runs',{})
        if name in runs:
            if group in ('pod','sr'):
                return group
            return runs[name]['kind']
    return 'unknown'

def get_r2(ytrue,ypred,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    ssres = ((ytrue-ypred)**2).sum(dim=dims,skipna=True)
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    return 1-ssres/sstot

def get_mse(ytrue,ypred,dims=None):
    dims = list(ytrue.dims) if dims is None else dims
    return ((ytrue-ypred)**2).mean(dim=dims,skipna=True)

def get_nn_complexity(kind,nfieldvars,nlevs,nlocalvars):
    def nparams(nfeatures):
        return (nfeatures*256)+256+(256*128)+128+(128*64)+64+(64*32)+32+(32*1)+1
    if kind=='baseline':
        return nparams(nfieldvars*nlevs+nlocalvars)
    elif kind=='nonparametric':
        return nfieldvars*nlevs+nparams(nfieldvars+nlocalvars)
    elif kind=='parametric':
        return 2*nfieldvars+nparams(nfieldvars+nlocalvars)

def get_mse_at_r2(ytrue,r2_target=0.5,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    n     = ytrue.count(dim=dims)
    return (1-r2_target)*sstot/n

def pareto_front(records):
    ordered      = sorted(records,key=lambda r: r['mse'])
    front        = []
    best_nparams = np.inf
    for r in ordered:
        if r['nparams'] < best_nparams:
            front.append(r)
            best_nparams = r['nparams']
    return front
def prettify_eq(eq_str):
    all_vars = ['rh','thetae','thetaestar','lf','shf','lhf']
    var_syms = {name: sp.Symbol(name) for name in all_vars}
    symbol_names = {
        var_syms['rh']:         r'\widehat{RH}',
        var_syms['thetae']:     r'\widehat{\theta_e}',
        var_syms['thetaestar']: r'\widehat{\theta_e^*}',
        var_syms['lf']:         r'\mathrm{LF}',
        var_syms['shf']:        r'\mathrm{SHF}',
        var_syms['lhf']:        r'\mathrm{LHF}'}
    ns = {**var_syms,
          'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
          'sqrt':sp.sqrt,'exp':sp.exp,'log':sp.log,'abs':sp.Abs,'cos':sp.cos,'sin':sp.sin}
    try:
        expr = eval(eq_str,{'__builtins__':{}},ns)
        for atom in list(expr.atoms(sp.Float)):
            expr = expr.subs(atom,sp.Float(f'{float(atom):.3g}'))
        return f'${sp.latex(expr,symbol_names=symbol_names)}$'
    except Exception:
        return eq_str


In [21]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

with xr.open_dataset(os.path.join(SPLITSDIR,'norm_train.h5'),engine='h5netcdf') as ds:
    firstvar = next(iter(MODELS['nn']['runs'].values()))['fieldvars'][0]
    nsigs    = ds.sizes['sig'] if 'sig' in ds[firstvar].dims else 1

results = {}
for group in ['pod','nn','sr']:
    for name in MODELS[group]['runs']:
        filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
        if not os.path.exists(filepath):
            continue
        with xr.open_dataset(filepath) as ds:
            predtp = ds.tp.load()
        if 'complexity' in predtp.dims:
            csvpath  = os.path.join(MODELSDIR,'sr',f'{name}_equations.csv')
            eqdf     = pd.read_csv(csvpath)
            best_c   = int(eqdf.loc[eqdf['loss'].idxmin(),'complexity'])
            predtp   = predtp.sel(complexity=best_c)
        elif 'seed' in predtp.dims:
            predtp = predtp.mean('seed')
        ytrue,ypred      = xr.align(truetp,predtp,join='inner')
        results[name]    = (ytrue,ypred)

print(f'Loaded {len(results)} models with predictions!')

Loaded 13 models with predictions!


In [ ]:
def plot_pareto():
    records      = []
    r2_half_mses = []
    for name,(ytrue,ypred) in results.items():
        mse = float(get_mse(ytrue,ypred))
        r2_half_mses.append(float(get_mse_at_r2(ytrue,r2_target=0.5)))
        kind = get_kind(name)
        if kind == 'pod':
            with np.load(os.path.join(MODELSDIR,'pod',f'{name}.npz')) as d:
                nparams = int(d['nparams'])
        elif kind == 'sr':
            csvpath = os.path.join(MODELSDIR,'sr',f'{name}_equations.csv')
            eqdf    = pd.read_csv(csvpath)
            nparams = int(eqdf.loc[eqdf['loss'].idxmin(),'complexity'])
        else:
            cfg     = MODELS['nn']['runs'][name]
            nparams = get_nn_complexity(cfg['kind'],len(cfg['fieldvars']),nsigs,len(cfg.get('localvars',[])))
        records.append(dict(name=name,mse=mse,nparams=nparams,kind=kind))
    fig,ax = pplt.subplots(refwidth=4,refheight=2.5)
    seen   = set()
    for rec in records:
        color,label = PLOTSTYLE.get(rec['kind'],('gray6',rec['kind']))
        ax.scatter(rec['mse'],rec['nparams'],color=color,marker='*',
                   s=60,zorder=3,label=label if label not in seen else None)
        seen.add(label)
    for rec in records:
        ax.annotate(rec['name'],(rec['mse'],rec['nparams']),xytext=(4,3),textcoords='offset points',fontsize=6)
    for i,mval in enumerate(sorted(set(np.round(r2_half_mses,12)))):
        ax.axvline(mval,color='k',linestyle=':',lw=1,label='$R^2=0.5$' if i==0 else None)
    front = pareto_front(records)
    if len(front) > 1:
        ax.plot([r['mse'] for r in front],[r['nparams'] for r in front],'k--',lw=1,label='Pareto front')
    ax.format(xlabel='MSE (mm$^2$)',xscale='log',xformatter='log',
              ylabel='Model Complexity',yscale='log',yformatter='log',grid=True)
    ax.legend(loc='r',ncols=1)
    pplt.show()

plot_pareto()

In [ ]:
def plot_pareto_equations():
    sr_runs = [name for name in SRCONFIG['runs']
        if os.path.exists(os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc'))]
    sr_runs = [name for name in sr_runs
               if 'complexity' in xr.open_dataset(os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')).dims]
    if not sr_runs:
        print('No SR Pareto predictions found (re-run evaluate.py to generate them)')
        return
    nruns  = len(sr_runs)
    fig,axes = pplt.subplots(ncols=nruns,refwidth=3.5,refheight=2.5,sharey=False)
    axes   = [axes] if nruns==1 else list(axes)
    for ax,name in zip(axes,sr_runs):
        with xr.open_dataset(os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')) as ds:
            predtp = ds.tp.load()
        csvpath = os.path.join(MODELSDIR,'sr',f'{name}_equations.csv')
        eqdf    = pd.read_csv(csvpath).set_index('complexity')
        ytrue_,predtp_ = xr.align(truetp,predtp,join='inner')
        complexities,mses,equations = [],[],[]
        for c in predtp_.coords['complexity'].values:
            if c not in eqdf.index:
                continue
            mse = float(get_mse(ytrue_,predtp_.sel(complexity=int(c))))
            complexities.append(int(c))
            mses.append(mse)
            equations.append(eqdf.loc[c,'equation'])
        color,_ = PLOTSTYLE['sr']
        ax.plot(complexities,mses,color=color,marker='o',ms=4,lw=1)
        for c,mse,eq in zip(complexities,mses,equations):
            ax.annotate(prettify_eq(eq),(c,mse),xytext=(4,3),textcoords='offset points',fontsize=5)
        ax.format(title=name,xlabel='Complexity',ylabel='MSE (mm$^2$)',grid=True)
    fig.format(suptitle='Pareto-Optimal SR Equations')
    pplt.show()

plot_pareto_equations()